# Clinical Dataset Analysis & Seaborn Visualizations
This notebook performs data cleaning, exploratory data analysis (EDA), and demographic & clinical outcome visualizations for the clinical dataset (`clinicaldata/clinical.tsv`) using **Seaborn** and **Matplotlib**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set Seaborn aesthetic theme
sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams["figure.figsize"] = (10, 6)

## 1. Load and Inspect Dataset
Replace standard clinical missing-value markers (`'--`, `--`, `not reported`) with `NaN`.

In [ ]:
na_tokens = ["'--", "--", "not reported", "Not Reported", "unknown", "Unknown"]
df = pd.read_csv("clinicaldata/clinical.tsv", sep="\t", low_memory=False, na_values=na_tokens)

print(f"Raw dataset dimensions: {df.shape[0]} rows, {df.shape[1]} columns")
df[["cases.case_id", "cases.disease_type", "cases.primary_site", "demographic.vital_status"]].head()

## 2. Patient Deduplication & Feature Engineering
Each case (patient) may have multiple rows for different diagnoses or treatment cycles. We deduplicate by `cases.case_id` for accurate patient-level metrics.

In [ ]:
# Patient-level cohort
df_patients = df.drop_duplicates(subset=["cases.case_id"]).copy()
print(f"Unique patient cohort count: {len(df_patients)}")

# Clean demographics and outcomes
df_patients["Age"] = pd.to_numeric(df_patients["demographic.age_at_index"], errors="coerce")
df_patients["Vital Status"] = df_patients["demographic.vital_status"].fillna("Unknown")
df_patients["Race"] = df_patients["demographic.race"].str.title().fillna("Not Reported")
df_patients["Sex"] = df_patients["demographic.sex_at_birth"].str.title().fillna("Not Reported")

# Simplify pathologic stage (Stage I, II, III, IV)
def simplify_stage(val):
    if not isinstance(val, str):
        return np.nan
    val = val.strip()
    if val.startswith("Stage I") and not val.startswith("Stage IV"):
        if val.startswith("Stage IA") or val.startswith("Stage IB") or val == "Stage I":
            return "Stage I"
        elif val.startswith("Stage IIA") or val.startswith("Stage IIB") or val == "Stage II":
            return "Stage II"
        elif val.startswith("Stage IIIA") or val.startswith("Stage IIIB") or val.startswith("Stage IIIC") or val == "Stage III":
            return "Stage III"
    elif val == "Stage IV":
        return "Stage IV"
    return np.nan

df_patients["Stage"] = df_patients["diagnoses.ajcc_pathologic_stage"].apply(simplify_stage)
df_patients[["cases.case_id", "Age", "Vital Status", "Race", "Stage"]].head()

## 3. Visualizations with Seaborn

In [ ]:
# Visual 1: Age Distribution by Vital Status (Histplot + KDE)
plt.figure(figsize=(9, 5))
sns.histplot(
    data=df_patients,
    x="Age",
    hue="Vital Status",
    kde=True,
    bins=25,
    element="step",
    palette={"Alive": "#2b5c8f", "Dead": "#d95f02"}
)
plt.title("Patient Age Distribution by Vital Status (Seaborn Histplot + KDE)", fontsize=13, fontweight="bold")
plt.xlabel("Age at Diagnosis (Years)")
plt.ylabel("Patient Count")
plt.tight_layout()
plt.show()

In [ ]:
# Visual 2: Survival / Vital Status Count
plt.figure(figsize=(6, 5))
ax = sns.countplot(
    data=df_patients,
    x="Vital Status",
    hue="Vital Status",
    palette={"Alive": "#2ca02c", "Dead": "#d62728"},
    legend=False
)
plt.title("Patient Survival Breakdown (Vital Status)", fontsize=13, fontweight="bold")
plt.xlabel("Vital Status")
plt.ylabel("Patient Count")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                ha="center", va="center", color="white", fontweight="bold", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Visual 3: AJCC Pathologic Stage Distribution
stage_order = ["Stage I", "Stage II", "Stage III", "Stage IV"]
stage_counts = df_patients["Stage"].value_counts().reindex(stage_order).dropna()

plt.figure(figsize=(8, 5))
ax = sns.barplot(
    x=stage_counts.index,
    y=stage_counts.values,
    hue=stage_counts.index,
    palette="Blues_r",
    legend=False
)
plt.title("AJCC Pathologic Stage Distribution", fontsize=13, fontweight="bold")
plt.xlabel("Pathologic Stage")
plt.ylabel("Patient Count")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2., p.get_height() + 5),
                ha="center", va="bottom", fontweight="bold", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Visual 4: Age Variation by Tumor Stage (Boxplot + Stripplot)
df_staged = df_patients[df_patients["Stage"].isin(stage_order)]

plt.figure(figsize=(8, 5))
sns.boxplot(
    data=df_staged,
    x="Stage",
    y="Age",
    order=stage_order,
    hue="Stage",
    palette="Set2",
    boxprops=dict(alpha=0.7),
    legend=False
)
sns.stripplot(
    data=df_staged,
    x="Stage",
    y="Age",
    order=stage_order,
    color="black",
    size=4,
    alpha=0.3,
    jitter=0.2
)
plt.title("Age Distribution Across Cancer Stages (Seaborn Boxplot + Stripplot)", fontsize=13, fontweight="bold")
plt.xlabel("Pathologic Stage")
plt.ylabel("Age (Years)")
plt.tight_layout()
plt.show()

In [ ]:
# Visual 5: Cohort Racial Representation
top_races = df_patients["Race"].value_counts().head(5)

plt.figure(figsize=(8, 4.5))
ax = sns.barplot(
    y=top_races.index,
    x=top_races.values,
    hue=top_races.index,
    palette="viridis",
    legend=False
)
plt.title("Patient Racial Demographics", fontsize=13, fontweight="bold")
plt.xlabel("Patient Count")
plt.ylabel("Race")
for p in ax.patches:
    val = int(p.get_width())
    ax.annotate(f"{val}", (val + 10, p.get_y() + p.get_height() / 2.),
                ha="left", va="center", fontweight="bold", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Visual 6: Common Treatment Regimens Administered
top_treatments = df["treatments.treatment_type"].dropna().value_counts().head(7)

plt.figure(figsize=(9, 5))
ax = sns.barplot(
    y=top_treatments.index,
    x=top_treatments.values,
    hue=top_treatments.index,
    palette="mako",
    legend=False
)
plt.title("Most Common Treatment Regimens Administered", fontsize=13, fontweight="bold")
plt.xlabel("Treatment Event Records")
plt.ylabel("Treatment Modality")
for p in ax.patches:
    val = int(p.get_width())
    ax.annotate(f"{val}", (val + 15, p.get_y() + p.get_height() / 2.),
                ha="left", va="center", fontweight="bold", fontsize=10)
plt.tight_layout()
plt.show()